# **Constraint Programming and Scheduling**

**Lecture (60 min) + lab (60 min)**  
**Setup:** Python 3; no external packages required.

By the end, you can model a small scheduling problem as a constraint satisfaction problem (CSP), use backtracking to find a solution, and explain how pruning avoids work.

| Time | Activity |
|---|---|
| 0–10 min | Variables, domains, and constraints |
| 10–25 min | Backtracking and pruning |
| 25–40 min | Worked map-coloring example |
| 40–60 min | Model a scheduling problem together |
| 60–120 min | Lab: schedule student presentations |

## 1. A scheduling problem is a CSP

A **constraint satisfaction problem** has:

- **Variables:** decisions to make (for example, the time assigned to each presentation).
- **Domains:** allowed values for each variable (for example, available time slots).
- **Constraints:** rules that a complete assignment must satisfy.

A **partial assignment** can already be rejected if it violates a constraint. This is **pruning**. **Backtracking** makes a choice, explores the consequences, and undoes that choice if it leads to a dead end.

For a schedule, a conflict constraint says two activities cannot use the same slot. Hard constraints must always hold; preferences can be optimized after we find feasible schedules.

## 2. Backtracking recipe

1. If every variable has a value, return the solution.
2. Pick an unassigned variable.
3. Try each value in its domain.
4. Keep a value only if it is consistent with the partial assignment.
5. Recurse; if it fails, undo the assignment and try the next value.

A simple improvement is to select the variable with the fewest currently legal values first (the **minimum-remaining-values** heuristic). We will use the basic version here so the search is easy to follow.

In [ ]:
def solve_csp(variables, domains, neighbors):
    assignment = {}

    def backtrack():
        if len(assignment) == len(variables):
            return assignment.copy()

        variable = next(v for v in variables if v not in assignment)
        for value in domains[variable]:
            if all(assignment.get(other) != value for other in neighbors[variable]):
                assignment[variable] = value
                result = backtrack()
                if result is not None:
                    return result
                del assignment[variable]
        return None

    return backtrack()

regions = ["A", "B", "C", "D"]
colors = {region: ["red", "blue", "green"] for region in regions}
neighboring_regions = {
    "A": ["B", "C"],
    "B": ["A", "C", "D"],
    "C": ["A", "B", "D"],
    "D": ["B", "C"],
}
solve_csp(regions, colors, neighboring_regions)

The neighbor rule is the constraint: adjacent regions must have different colors. The consistency check is performed **before** recursion, so an invalid partial assignment is never explored further. This same pattern works for exam timetables, staff shifts, and room allocation.

### Check your understanding

1. In the presentation problem, what are the variables and domains?
2. Which assignments can be rejected before the schedule is complete?
3. What does the search return if no assignment satisfies all constraints?

**Instructor prompt:** ask students to name one real constraint that is hard and one that is only a preference.

## Lab (60 min): schedule student presentations

Four students (`Ada`, `Ben`, `Cy`, `Dee`) must present in one of three time slots (`9:00`, `10:00`, `11:00`). Students who are working together must present at different times. The conflict pairs are `Ada–Ben`, `Ben–Cy`, and `Cy–Dee`.

### Tasks

1. Define the variables, domains, and conflict constraints in Python.
2. Complete the `is_consistent` function so a student cannot share a slot with an already assigned partner.
3. Complete the recursive search. Return a dictionary for a valid schedule, or `None` if there is no solution.
4. Run the supplied checks and add one extra conflict pair of your choice. Does the schedule change?
5. In two sentences, explain where pruning happens and why checking partial assignments is useful.

**Suggested timing:** model (10 min), implement (25 min), test and modify (15 min), explain (10 min).

In [ ]:
students = ["Ada", "Ben", "Cy", "Dee"]
slots = ["9:00", "10:00", "11:00"]
domains = {student: slots[:] for student in students}
conflicts = {
    "Ada": ["Ben"],
    "Ben": ["Ada", "Cy"],
    "Cy": ["Ben", "Dee"],
    "Dee": ["Cy"],
}

def is_consistent(student, slot, assignment):
    # TODO: return False if an already scheduled partner has this slot.
    raise NotImplementedError

def search(assignment):
    # TODO: return a complete solution, or None if all choices fail.
    # Hint: use recursion and remove a tentative assignment after failure.
    raise NotImplementedError

schedule = search({})
print(schedule)

In [ ]:
# Run after completing the starter code.
assert schedule is not None
assert set(schedule) == set(students)
assert all(schedule[a] != schedule[b] for a, partners in conflicts.items() for b in partners)
print("All scheduling constraints are satisfied.")

### Exit ticket

Submit your notebook with the completed code, passing checks, the schedule you found, and your short explanation of pruning. **Extension:** add a fourth slot, or add a preference (such as Ada preferring 10:00) and discuss how optimization differs from finding any feasible schedule.